# Non-linear timeseries

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from filtering import (
    particle_filter,
    particle_mean,
    particle_std,
    simulate,
)

In [ ]:
T = 100
dt_sim = 1.0
dt_obs = 1.0

D = 1
O = 1


Q = 10.0 * jnp.eye(D)
R = 1.0 * jnp.eye(O)

In [ ]:
def f(x, t, dt):
    return 0.5 * x + 25.0 * x / (1 + jnp.square(x)) + 8.0 * jnp.cos(1.2 * t)


def h(x):
    return x

In [ ]:
key = jax.random.PRNGKey(seed=17)

x0 = jnp.array([0.0])

(t_sim, x_sim), (t_obs, y_obs) = simulate(f, h, x0, T, dt_sim, dt_obs, Q, R, key)

In [ ]:
N = 10_000

particles, log_weights, ess, marginal_log_like = particle_filter(
    f,
    h,
    t_obs,
    y_obs,
    n_particles=N,
    x0_mean=jnp.zeros(D),
    x0_cov=jnp.eye(D),
    Q=Q,
    R=R,
    key=key,
    n_substeps=5,
)

In [ ]:
mean = particle_mean(particles, log_weights)
std = particle_std(particles, log_weights)

fig, axs = plt.subplots(D + 1, 1, figsize=(8, 2.5 * (D + 1)), sharex=True)
for i in range(D):
    axs[i].plot(t_sim, x_sim[..., i], lw=0.8, alpha=0.7)
    axs[i].plot(t_obs, mean[..., i], lw=1.5, alpha=0.7)
    axs[i].fill_between(
        t_obs,
        mean[..., i] - 2 * std[..., i],
        mean[..., i] + 2 * std[..., i],
        alpha=0.3,
        step="mid",
    )
    axs[i].set_ylabel(f"x{i + 1}")
    axs[i].grid()
for j in range(O):
    axs[j].scatter(t_obs, y_obs[..., j], marker="x", s=10.0, alpha=0.7, c="r")
axs[-1].plot(t_obs, ess)
axs[-1].axhline(y=0.5 * N, ls="--", c="black", alpha=0.3)
axs[-1].set_ylim(0, 1.1 * N)
axs[-1].set_ylabel("ESS")
axs[-1].set_xlabel("t (s)")
axs[-1].grid(True)
plt.tight_layout();